# Catchment analysis of networks

## Step 0: Import packages and set up base paths and crs

### Import packages

In [ ]:
from pathlib import Path
import fiona
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import os
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from IPython.display import display

### Set up base paths and crs

#### Base path and network path

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
networks_path = base_path / "Processed_data/networks"

#### Output path

In [ ]:
output_folder = base_path / "Outputs"  # or specify another folder

#### CRS

In [ ]:
jamaica_metric_grid_crs = "EPSG:3448"

## Step 1: Read in useful data

### Hydrobasins

In [ ]:
hydrobasins = base_path / "Processed_data/HydroBASINS_Level12_Clipped_Jamaica.shp"
hydrobasins = gpd.read_file(hydrobasins)
print(hydrobasins.crs)

In [ ]:
jamaica_boundary_path = base_path / "Inputs/Boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(f"Original Jamaica boundary CRS: {jamaica_boundary.crs}")

### Networks

In [ ]:
network_layers = pd.read_csv(base_path / "Processed_data" / "network_layers_hazard_intersections_details.csv")
network_layers

In [ ]:
network_layers.columns

#### Create a dictionary of the network data and in this notebook's memory change the node id and edge id column to asset_id

In [ ]:
network_data = {}

for row in network_layers.itertuples():
    gpkg_fname = networks_path / ".." / row.path
    layer = row.asset_layer
    asset_id_column = row.asset_id_column  # Retrieve the asset_id_column
    key = f"{gpkg_fname.stem}_{layer}"
    print(f"Loading {key}")

    try:
        data = gpd.read_file(gpkg_fname, layer=layer)

        # Reproject to Jamaica's CRS if necessary
        if data.crs != jamaica_metric_grid_crs:
            data = data.to_crs(jamaica_metric_grid_crs)

        # Ensure asset_id_column exists in the data
        if asset_id_column in data.columns:
            # Rename the asset ID column to a consistent name: 'asset_id'
            data = data.rename(columns={asset_id_column: "asset_id"})
            print(f"  Renamed column '{asset_id_column}' to 'asset_id' for {key}.")       
        else:
            # Handle the case where the asset ID column is missing
            print(f"  WARNING: Column '{asset_id_column}' not found in {key}.")

        # Store the processed data
        network_data[key] = data
        print(f"Loaded and reprojected: {gpkg_fname.name}, layer {layer}")
    except Exception as e:
        print(f"Failed to load {gpkg_fname.name}:{layer} {e}")

# Display confirmation
print(f"Successfully loaded {len(network_data)} network layers.")

# Debugging: Check if 'asset_id' exists in all layers
print("\nChecking 'asset_id' column presence in all layers...")
for key, gdf in network_data.items():
    print(f"{key}: 'asset_id' column exists: {'asset_id' in gdf.columns}")

#### Can have a look at the dictionary by displaying it

In [ ]:
# display(network_data['rail_edges'])
display(network_data)

##  National analysis

### Configuration for sectors and subsectors

In [ ]:
sector_config = {
    "Transport": {
        "Rail": {
            "layers": [
                {"layer": "rail_edges", "metric": "count", "group_by": "asset_type"},
                {"layer": "rail_nodes", "metric": "count", "group_by": "asset_type"}
            ]
        },
        "Roads": {
            "layers": [
                {"layer": "roads_edges", "metric": "length", "group_by": "asset_type"},
                {"layer": "roads_nodes", "metric": "count", "group_by": "asset_type"}
            ]
        },
        "Airports": {
            "layers": [
                {"layer": "airport_polygon_areas", "metric": "count", "filter": {"asset_type": "terminal"}},
                {"layer": "airport_polygon_areas", "metric": "count", "filter": {"asset_type": "runway"}}
            ]
        },
        "Ports": {
            "layers": [
                {"layer": "port_polygon_areas", "metric": "count"}
            ]
        },
    },
    "Electricity": {
        "Electricity Network": {
            "layers": [
                {"layer": "electricity_network_v3.1_nodes", "metric": "count", "filter": {"asset_type": "source"}},
                {"layer": "electricity_network_v3.1_nodes", "metric": "count", "filter": {"asset_type": "junction"}},
                {"layer": "electricity_network_v3.1_nodes", "metric": "count", "filter": {"asset_type": "sink"}},
                {"layer": "electricity_network_v3.1_edges", "metric": "length", "filter": {"asset_type": "High Voltage"}},
                {"layer": "electricity_network_v3.1_edges", "metric": "length", "filter": {"asset_type": "Low Voltage"}}
            ]
        },
    },
    "Water": {
        "Pipelines": {
            "layers": [
                {"layer": "pipelines_NWC_edges", "metric": "length"}
            ]
        },
        "Facilities": {
            "layers": [
                {"layer": "waste_water_facilities_NWC_nodes", "metric": "count"},
                {"layer": "irrigation_assets_NIC_nodes", "metric": "count"},
                {"layer": "irrigation_assets_NIC_edges", "metric": "count"},
                {"layer": "potable_facilities_NWC_nodes", "metric": "count"}
            ]
        },
    },
    "Buildings": {
        "Buildings": {
            "layers": [
                {"layer": "buildings_assigned_economic_activity_areas", "metric": "count", "group_by": "building_type"}
            ]
        },
    },
}

### Code for summarizing at national level 

In [ ]:
def summarize_national_level_with_asset_id_check(network_data, config):
  
    national_summaries = []

    for sector, subsectors in config.items():
        print(f"Processing Sector: {sector}")
        for subsector, details in subsectors.items():
            layers = details.get("layers", [])

            for layer_entry in layers:
                # Retrieve layer details
                layer_name = layer_entry["layer"]
                metric = layer_entry["metric"]
                group_by = layer_entry.get("group_by", [])
                filters = layer_entry.get("filter", {})

                # Ensure group_by is a list
                if isinstance(group_by, str):
                    group_by = [group_by]
                elif not group_by:
                    group_by = []

                # Check if the layer exists in the network data
                if layer_name not in network_data:
                    print(f"  Layer {layer_name} not found in network data. Skipping.")
                    continue

                gdf = network_data[layer_name]
                print(f"  Processing Layer: {layer_name}, Features: {len(gdf)}")

                # Apply filters if provided
                filtered_gdf = gdf.copy()
                if filters:
                    print(f"    Applying Filters: {filters}")
                    for column, value in filters.items():
                        if column not in filtered_gdf.columns:
                            print(f"    WARNING: filter column '{column}' not found in {layer_name} columns. Skipping that filter.")
                            
                        else:
                            filtered_gdf = filtered_gdf[filtered_gdf[column] == value]
                    print(f"    Remaining Features: {len(filtered_gdf)} after filtering")

                # Debug: check if there's data left
                if filtered_gdf.empty:
                    print(f"    No data matches the filter for {layer_name}. Skipping.")
                    continue

                # Debug: Show group_by columns vs. actual columns
                print(f"    DEBUG: group_by = {group_by}")
                print(f"    DEBUG: filtered_gdf columns = {filtered_gdf.columns.tolist()}")

                # Summarize based on metrics
                if metric == "length":
                    print(f"    Summarizing length for {layer_name}.")
                    filtered_gdf["length_km"] = filtered_gdf.geometry.length / 1000
                    try:
                        grouped = filtered_gdf.groupby(group_by)["length_km"].sum().reset_index()
                        grouped.rename(columns={"length_km": "Total Length (km)"}, inplace=True)
                    except ValueError as ve:
                        print(f"    ERROR during groupby with {group_by}: {ve}")
                        continue

                elif metric == "count":
                    print(f"    Summarizing count for {layer_name}.")
                    try:
                        grouped = filtered_gdf.groupby(group_by).size().reset_index(name="Feature Count")
                    except ValueError as ve:
                        print(f"    ERROR during groupby with {group_by}: {ve}")
                        continue
                else:
                    print(f"    Unsupported metric: {metric}")
                    continue

                # Add metadata and append the summary
                grouped["Sector"] = sector
                grouped["Subsector"] = subsector
                grouped["Layer"] = layer_name
                national_summaries.append(grouped)

    # Combine all summaries into a single DataFrame
    if national_summaries:
        combined_summaries = pd.concat(national_summaries, ignore_index=True)
        return combined_summaries
    else:
        print("No valid summaries were created.")
        return pd.DataFrame()

### National level analysis summary

In [ ]:
national_summaries = summarize_national_level_with_asset_id_check(network_data, sector_config)

if not national_summaries.empty:
    print("\nNational-Level Summaries:")
    display(national_summaries)
else:
    print("No valid summaries were created for national level.")

## Step 6: Catchment analysis

#### Define sector and subsector-specific output directory dynamically

In [ ]:
def get_output_directory(sector, subsector):
    output_directory = os.path.join("catchment_outputs", sector, subsector)
    os.makedirs(output_directory, exist_ok=True)
    return output_directory

#### Summarize infra data by catchment, saving intersected data as GPKG.

In [ ]:
def summarize_and_save_by_catchment(
    gdf, 
    hydrobasins, 
    layer_name, 
    catchment_id_col="HYBAS_ID", 
    group_by=None, 
    sector=None, 
    subsector=None
):
    
    if gdf.empty:
        print(f" Layer '{layer_name}' is empty. Skipping.")
        return pd.DataFrame(columns=["Layer", "Catchment_ID", "Metric_Value"])

    group_by = group_by if isinstance(group_by, list) else [group_by] if group_by else []

    # # Reproject if needed
    # if gdf.crs != hydrobasins.crs:
    #     print(f"  Reprojecting '{layer_name}' to match HydroBASINS CRS...")
    #     gdf = gdf.to_crs(hydrobasins.crs)

    # Figure out geometry type
    geom_types = gdf.geom_type.unique()
    is_line = any("LineString" in gt for gt in geom_types)
    is_polygon = any("Polygon" in gt for gt in geom_types)
    is_point = any("Point" in gt for gt in geom_types)

    # Output path
    output_directory = get_output_directory(sector, subsector)
    output_path = os.path.join(output_directory, f"{layer_name}_with_hydrobasins.gpkg")

    # Fix invalid geoms
    invalid_geoms = gdf[~gdf.is_valid]
    if not invalid_geoms.empty:
        print(f"  Found {len(invalid_geoms)} invalid geometries. Buffering (fixing) them.")
        gdf['geometry'] = gdf['geometry'].buffer(0)

    # Remove empty geoms
    empty_geoms = gdf[gdf.geometry.is_empty]
    if not empty_geoms.empty:
        print(f"  Found {len(empty_geoms)} empty geometries. Removing them.")
        gdf = gdf[~gdf.geometry.is_empty]

    # Process lines
    if is_line:
        print(f"Processing '{layer_name}' as line data...")
        if "asset_id" not in gdf.columns:
            print(f"  WARNING: 'asset_id' not in columns; adding placeholder.")
            gdf["asset_id"] = gdf.index.astype(str)

        try:
            infra_in_basin = gpd.overlay(gdf, hydrobasins, how="identity")
            print(f"  Overlaid lines. {len(infra_in_basin)} features created.")
        except Exception as e:
            print(f"  ERROR during overlay for '{layer_name}': {e}")
            return pd.DataFrame()

        infra_in_basin.to_file(output_path, driver="GPKG")
        infra_in_basin["Metric_Value"] = infra_in_basin.geometry.length / 1000
        metric_col_name = "Total Length (km)"

    elif is_polygon or is_point:
        print(f"Processing '{layer_name}' as polygon/point data...")
        try:
            infra_in_basin = gpd.sjoin(gdf, hydrobasins, how="inner", predicate="intersects")
            print(f"  Spatially joined. {len(infra_in_basin)} features created.")
        except Exception as e:
            print(f"  ERROR in sjoin for '{layer_name}': {e}")
            return pd.DataFrame()

        infra_in_basin.to_file(output_path, driver="GPKG")

        if is_polygon:
            infra_in_basin["Metric_Value"] = infra_in_basin.geometry.area / 1e6
            metric_col_name = "Total Area (km²)"
        elif is_point:
            infra_in_basin["Metric_Value"] = 1
            metric_col_name = "Feature Count"

    # Summarize by HYBAS_ID
    if group_by:
        grouped = infra_in_basin.groupby([catchment_id_col] + group_by)["Metric_Value"].sum().reset_index()
    else:
        grouped = infra_in_basin.groupby(catchment_id_col)["Metric_Value"].sum().reset_index()

    grouped = grouped.rename(columns={"Metric_Value": metric_col_name})
    grouped["Layer"] = layer_name
    return grouped

#### Summarize all infrastructure layers by catchment & save outputs.

In [ ]:
def summarize_and_save_catchment_level(
    network_data, 
    config, 
    hydrobasins, 
    catchment_id_col="HYBAS_ID"
):
    
    catchment_summaries = []

    for sector, subsectors in config.items():
        print(f"\nProcessing Sector: {sector}")
        for subsector, details in subsectors.items():
            layers = details.get("layers", [])

            for layer_entry in layers:
                layer_name = layer_entry["layer"]
                group_by = layer_entry.get("group_by", [])
                filters = layer_entry.get("filter", {})

                gdf = network_data.get(layer_name)
                if gdf is None or gdf.empty:
                    print(f"  Layer '{layer_name}' is missing or empty. Skipping.")
                    continue

                print(f"Processing Layer: {layer_name}, {len(gdf)} features")

                # Apply filters
                if filters:
                    print(f"  Filters: {filters}")
                    for col, val in filters.items():
                        if col in gdf.columns:
                            gdf = gdf[gdf[col] == val]
                    print(f"   After filters: {len(gdf)} features remain")

                if gdf.empty:
                    print(f"   No data after filtering. Skipping '{layer_name}'.")
                    continue

                summary = summarize_and_save_by_catchment(
                    gdf=gdf,
                    hydrobasins=hydrobasins,
                    layer_name=layer_name,
                    catchment_id_col=catchment_id_col,
                    group_by=group_by,
                    sector=sector,
                    subsector=subsector
                )

                if not summary.empty:
                    summary["Sector"] = sector
                    summary["Subsector"] = subsector
                    catchment_summaries.append(summary)

    if catchment_summaries:
        return pd.concat(catchment_summaries, ignore_index=True)
    else:
        print("No valid catchment summaries were created.")
        return pd.DataFrame()

In [ ]:
catchment_summaries = summarize_and_save_catchment_level(
    network_data, 
    sector_config, 
    hydrobasins, 
    catchment_id_col="HYBAS_ID"
)

if not catchment_summaries.empty:
    print("\nCatchment-Level Summaries:")
    display(catchment_summaries)
else:
    print("No valid summaries were created at the catchment level.")

In [ ]:
robyns_cities = ['London','Liverpool','Oxford']; robyns_dinners_dict = {0: 'Gnocchi', 1: 'Tacos', 2: 'Pasta', 3: 'Unknown', 4:"Duck", 5:"Unknown", 6:"Unknown"}

In [ ]:
robyns_dinners_dict[2]

In [ ]:
robyns_dinners_dict